# Surrogate-assisted (Bayesian) multi-objective optimization with a GPR emulator

In the previous MOU notebook we ran PESTPP-MOU's NSGA-II directly against the *numerical* Freyberg model. Every individual in every generation is a full model run, so the cost is `population_size x generations` true model runs - thousands of them.

Here we do something different: **surrogate-assisted (Bayesian) multi-objective optimization** using a **Gaussian Process Regression (GPR)** emulator from `pyemu.emulators`. The idea is an **inner-outer loop**:

- **Outer loop (expensive, few runs):** run the *true* model on a small batch of decision-variable sets, then (re)train a GPR surrogate that maps **decision variables -> objectives**. The GP gives both a mean prediction *and* a predictive uncertainty for every objective.
- **Inner loop (cheap, many runs):** run a full PESTPP-MOU evolutionary search **against the GPR emulator** instead of the model. Because an emulator evaluation is a couple of matrix operations, we can afford a whole NSGA-II run for almost nothing. This produces an **emulated Pareto population**.
- **Infill:** pick a handful of promising / uncertain members from that inner emulated population, evaluate them on the *true* model, add them to the training set, and go around again.

This is the multi-objective flavour of **Bayesian optimization** / efficient global optimization: spend the expensive true-model budget only where the surrogate says the action is (near the trade-off front) or where the surrogate is most unsure.

> **Note:** this notebook uses the `GPR` emulator, which is currently only in the local development version of `pyemu`. The first code cell points at that checkout.

Let'z'go!

## The inner-outer scheme at a glance

The cell below draws the loop we are going to implement. Read it top to bottom: the **outer** loop (blue) touches the expensive true model; the **inner** loop (orange) lives entirely inside the cheap emulator.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, shutil, time
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

# >>> point at the local (dev) pyemu that ships the GPR emulator <<<
local_pyemu = os.path.abspath(os.path.join("..","..","..","pyemu"))
if os.path.exists(local_pyemu):
    sys.path.insert(0, local_pyemu)
import pyemu
from pyemu.emulators import GPR
print("using pyemu from:", pyemu.__file__)
sys.path.insert(0,"..")
import herebedragons as hbd

In [ ]:
def draw_scheme():
    fig, ax = plt.subplots(1, 1, figsize=(9.5, 6.5))
    ax.axis("off")
    def box(cx, cy, w, h, text, fc):
        ax.add_patch(mpatches.FancyBboxPatch((cx-w/2, cy-h/2), w, h,
                     boxstyle="round,pad=0.015", fc=fc, ec="k", lw=1.5, alpha=0.95))
        ax.text(cx, cy, text, ha="center", va="center", fontsize=10)
    def arrow(a, b, color="0.3", rad=0.0):
        ax.add_patch(FancyArrowPatch(a, b, arrowstyle="-|>", mutation_scale=18,
                     lw=1.7, color=color, connectionstyle=f"arc3,rad={rad}"))
    BL, OR, GR = "#bcd6f0", "#f7d29a", "#d7e9c9"
    box(0.45,0.90,0.42,0.09,"initial design:\nsample decision variables", BL)
    box(0.45,0.74,0.42,0.09,"TRUE model runs (expensive)", BL)
    box(0.45,0.58,0.42,0.10,"train / update GPR emulator\n(dec vars -> objectives, + uncertainty)", GR)
    box(0.25,0.36,0.40,0.13,"INNER loop:\nPESTPP-MOU on the\nGPR emulator (cheap)", OR)
    box(0.70,0.36,0.34,0.13,"emulated\nPareto population", OR)
    box(0.45,0.14,0.42,0.09,"select INFILL points\n(spread + most-uncertain)", BL)
    arrow((0.45,0.855),(0.45,0.79))                 # initial -> true
    arrow((0.45,0.695),(0.45,0.63))                 # true -> train
    arrow((0.40,0.53),(0.30,0.425))                 # train -> inner
    arrow((0.45,0.36),(0.53,0.36))                  # inner -> emulated pop
    arrow((0.70,0.295),(0.52,0.18))                 # emulated pop -> infill
    # outer feedback: infill -> true model, routed up the left side
    ax.add_patch(FancyArrowPatch((0.24,0.14),(0.08,0.14), arrowstyle="-", lw=1.7, color="#1f6fb4"))
    ax.add_patch(FancyArrowPatch((0.08,0.14),(0.08,0.74), arrowstyle="-", lw=1.7, color="#1f6fb4"))
    arrow((0.08,0.74),(0.24,0.74), color="#1f6fb4")
    ax.text(0.055,0.44,"OUTER loop\n(infill -> true model)", rotation=90,
            color="#1f6fb4", fontsize=10, va="center", ha="center")
    ax.text(0.475,0.36,"  <->  ", fontsize=12, ha="center", va="center", color="0.3")
    ax.text(0.475,0.30,"inner loop", fontsize=9, ha="center", va="center", color="#b5651d")
    ax.set_xlim(0,1); ax.set_ylim(0.05,1)
    fig.tight_layout()
    return fig

_ = draw_scheme()

## The optimization problem

We re-use the exact Freyberg management problem from the ["freyberg mou"](freyberg_mou_1.ipynb) notebook and the optimization template built in [part2_08_opt](../part2_08_opt/freyberg_opt_1.ipynb). The **decision variables** are the groundwater extraction rates (the `decvars` parameter group). There are two competing **objectives**, both minimized:

- `cum ... sfr` - cumulative groundwater/surface-water exchange (stream impact)
- `cum ... wel` - cumulative well extraction (negative; minimizing it = extracting more)

To keep the emulator focused and the true-model sweeps light, we trim the control file down to just the two objective observations and drop the prior-information equations (the surrogate maps decision variables straight to the two objectives).

In [ ]:
t_d = "gpr_bo_template"
if os.path.exists(t_d): shutil.rmtree(t_d)
org_t_d = os.path.join("..","part2_08_opt","freyberg6_template")
if not os.path.exists(org_t_d):
    raise Exception("you need to run the part2_08_opt/freyberg_opt_1.ipynb notebook")
shutil.copytree(org_t_d, t_d)

pst_true = pyemu.Pst(os.path.join(t_d,"pest.pst"))
par_true = pst_true.parameter_data
# fix everything, free only the decision variables
par_true["partrans"] = "fixed"
dvnames = par_true.loc[par_true.pargp=="decvars","parnme"].tolist()
par_true.loc[dvnames,"partrans"] = "none"
dvlb = par_true.loc[dvnames,"parlbnd"].values.astype(float)
dvub = par_true.loc[dvnames,"parubnd"].values.astype(float)

OBJS = ["oname:cum_otype:lst_usecol:sfr_totim:4383.5",
        "oname:cum_otype:lst_usecol:wel_totim:4383.5"]
OBJ_LABELS = {OBJS[0]:"cum sfr exchange", OBJS[1]:"cum wel extraction"}

# trim observations down to just the objective group (keep only cum.csv.ins)
for ins in list(pst_true.instruction_files):
    if os.path.basename(ins) != "cum.csv.ins":
        pst_true.drop_observations(os.path.join(t_d,ins), pst_path=".")
obs = pst_true.observation_data
obs["weight"] = 0.0
obs.loc[OBJS,"weight"] = 1.0
obs.loc[OBJS,"obgnme"] = "less_than_obj"   # both objectives are minimized
pst_true.prior_information = pst_true.null_prior
print("number of decision variables:", len(dvnames))
print("number of (trimmed) observations:", pst_true.nobs)

## The true-model "oracle"

We need a way to push a *batch* of decision-variable sets through the real model and collect the two objectives. PESTPP-SWP (the parametric sweep tool) is exactly that. We write one control file configured for a sweep and a small helper that runs it with parallel workers and returns the objectives.

In [ ]:
# write a sweep-configured control file
pst_swp = pst_true   # same object; we just set sweep options
pst_swp.pestpp_options = {}
pst_swp.pestpp_options["sweep_parameter_csv_file"] = "sweep_in.csv"
pst_swp.pestpp_options["sweep_output_csv_file"] = "sweep_out.csv"
pst_swp.control_data.noptmax = 0
pst_swp.write(os.path.join(t_d,"pest_swp.pst"), version=2)

NUM_WORKERS = 12   # set according to your machine
true_run_count = {"n": 0}

def evaluate_true(dvdf, tag):
    """run the TRUE model on a population of decision-variable sets via pestpp-swp.
    dvdf: DataFrame with the decision-variable columns. returns a copy with the
    two objective columns appended (raw model units)."""
    base = par_true["parval1"].astype(float)
    sw = pd.DataFrame(np.repeat(base.values.reshape(1,-1), dvdf.shape[0], axis=0),
                      columns=base.index)
    sw.loc[:, dvnames] = dvdf[dvnames].values
    sw.index = [str(i) for i in range(dvdf.shape[0])]
    sw.index.name = "real_name"
    sw.to_csv(os.path.join(t_d,"sweep_in.csv"))
    m_d = "master_swp_"+tag
    if os.path.exists(m_d): shutil.rmtree(m_d)
    pyemu.os_utils.start_workers(t_d,"pestpp-swp","pest_swp.pst",num_workers=NUM_WORKERS,
                                 worker_root=".", master_dir=m_d)
    out = pd.read_csv(os.path.join(m_d,"sweep_out.csv"), index_col=0)
    res = dvdf[dvnames].copy().reset_index(drop=True)
    for o in OBJS:
        res[o] = out.loc[:, o].values
    true_run_count["n"] += res.shape[0]
    shutil.rmtree(m_d)
    return res

## Configuration

The most important knob is **`N_OUTER`** - the number of outer infill cycles. Each cycle does one inner emulated MOU search and one batch of true-model infill runs. Crank it up to spend more true-model budget refining the front; turn it down for a quicker run.

In [ ]:
# ============================ user settings ============================
N_OUTER    = 3      # <-- number of OUTER infill cycles (the main knob)
N_INIT     = 100    # size of the initial space-filling design (true-model runs)
N_INFILL   = 20     # true-model infill runs per outer cycle
INNER_POP  = 50     # population size for the inner (emulated) MOU search
INNER_GENS = 25     # generations for the inner (emulated) MOU search
SEED       = 22220
# ======================================================================
rng = np.random.default_rng(SEED)
print(f"plan: {N_INIT} initial true runs, then {N_OUTER} cycles x {N_INFILL} infill runs")
print(f"  ~= {N_INIT + N_OUTER*N_INFILL} true-model runs total")
print(f"  each cycle runs an emulated MOU of {INNER_POP} x {INNER_GENS} = {INNER_POP*INNER_GENS} FREE emulator evals")

## Initial design

Draw a space-filling (uniform) sample of decision-variable sets and evaluate them on the true model. This is the only "big" batch of true runs; everything after is small, targeted infill.

In [ ]:
init_dv = pd.DataFrame(rng.uniform(dvlb, dvub, size=(N_INIT, len(dvnames))), columns=dvnames)
print("evaluating initial design on the true model...")
train = evaluate_true(init_dv, "init")
train["cycle"] = 0
print("training set:", train.shape, " true runs so far:", true_run_count["n"])
train[OBJS].describe().loc[["mean","std","min","max"]]

## The GPR emulator

A couple of important details for fitting a GP surrogate to objectives that live at ~1e7:

1. **Standardize the objectives.** The GP only standard-scales its *inputs*; the targets are left raw. With targets of magnitude ~1e7 the default kernel can't represent them and the surrogate generalizes terribly. Standardizing each objective to zero mean / unit variance fixes this (held-out R^2 jumps from ~0.7 to ~0.98). We freeze the scaling from the initial design so the surrogate's target space doesn't drift between cycles.
2. **Keep the predictive uncertainty.** `return_std=True` gives us a per-objective standard deviation that we'll use to drive the "explore" part of infill.

In [ ]:
# freeze objective standardization from the initial design
obj_mean = train[OBJS].mean()
obj_std  = train[OBJS].std()

def standardize(df):
    d = df.copy()
    d[OBJS] = (df[OBJS] - obj_mean) / obj_std
    return d

def destd_val(arr, o):     # standardized -> raw objective units
    return arr * obj_std[o] + obj_mean[o]

def fit_gpr(train_raw):
    data = standardize(train_raw[dvnames + OBJS])
    gpr = GPR(data=data, input_names=dvnames, output_names=OBJS,
              transforms=[{"type":"standard_scaler"}], n_restarts_optimizer=5,
              return_std=True, verbose=False)
    gpr.fit()
    return gpr

print("fitting initial GPR emulator...")
gpr = fit_gpr(train)
# quick in-sample sanity check
pr = gpr.predict(train[dvnames])
for o in OBJS:
    yhat = destd_val(pr[o].values, o)
    ss = ((train[o]-train[o].mean())**2).sum()
    r2 = 1 - ((train[o]-yhat)**2).sum()/ss
    print(f"  {OBJ_LABELS[o]:22s} in-sample R2={r2:.4f}")

## The inner loop: PESTPP-MOU on the emulator

`gpr.prepare_pestpp(...)` writes a complete PEST++ interface whose "model" is the GPR emulator: its parameters are the decision variables, its observations are the two objectives (plus a `*_gprstd` uncertainty observation for each). We then point PESTPP-MOU at it and let NSGA-II evolve a population - but every evaluation is an emulator call, not a model run.

To make those emulator calls fast we run them **in memory** with `pyemu.helpers.gpr_pyworker`: each parallel worker holds the fitted GPR object and answers PEST++ directly, so there is no per-run process spin-up or disk I/O. The result is the **emulated Pareto population** - the inner archive, returned here with destandardized objectives and the emulator's predictive std for each member.

In [ ]:
def inner_mou(gpr, cycle):
    gt_d = "emu_template"
    pst_g = gpr.prepare_pestpp(gt_d, pst=pst_true)
    for f in os.listdir(t_d):
        if f.startswith("pestpp-mou"):
            shutil.copy2(os.path.join(t_d,f), os.path.join(gt_d,f))
    # drop any optimization/forecast options inherited from the source pst
    for k in ["forecasts","predictions","opt_dec_var_groups","opt_objective_function",
              "opt_constraint_groups","base_jacobian","hotstart_resfile"]:
        pst_g.pestpp_options.pop(k, None)
    g = pst_g.observation_data
    g.loc[OBJS,"weight"] = 1.0
    g.loc[OBJS,"obgnme"] = "less_than_obj"
    pst_g.prior_information = pst_g.null_prior
    pst_g.pestpp_options["mou_objectives"] = OBJS
    pst_g.pestpp_options["mou_population_size"] = INNER_POP
    pst_g.pestpp_options["mou_save_population_every"] = INNER_GENS
    pst_g.pestpp_options["mou_generator"] = "de"
    # random initial population for the inner search
    ip = pd.DataFrame(rng.uniform(dvlb,dvub,size=(INNER_POP,len(dvnames))), columns=dvnames)
    ip.index = [str(i) for i in range(INNER_POP)]; ip.index.name = "real_name"
    ip.to_csv(os.path.join(gt_d,"inner_init.csv"))
    pst_g.pestpp_options["mou_dv_population_file"] = "inner_init.csv"
    pst_g.control_data.noptmax = INNER_GENS
    pst_g.write(os.path.join(gt_d,"gpr.pst"), version=2)

    gm_d = "emu_master"
    if os.path.exists(gm_d): shutil.rmtree(gm_d)
    input_df = pd.read_csv(os.path.join(gt_d,"emulator_input.csv"), index_col=0)
    # run the inner MOU with the emulator evaluated IN MEMORY (no disk i/o)
    pyemu.os_utils.start_workers(gt_d,"pestpp-mou","gpr.pst",num_workers=NUM_WORKERS,
                                 worker_root=".", master_dir=gm_d,
                                 ppw_function=pyemu.helpers.gpr_pyworker,
                                 ppw_kwargs={"gpr":gpr, "input_df":input_df})
    adv = pd.read_csv(os.path.join(gm_d,"gpr.archive.dv_pop.csv"), index_col=0)
    aob = pd.read_csv(os.path.join(gm_d,"gpr.archive.obs_pop.csv"), index_col=0)
    front = adv.loc[:, dvnames].copy()
    for o in OBJS:
        front[o] = destd_val(aob.loc[front.index, o].values, o)        # emulated objective
        s = o + "_gprstd"
        if s in aob.columns:
            front[s] = aob.loc[front.index, s].values * obj_std[o]      # emulator uncertainty
    shutil.rmtree(gm_d); shutil.rmtree(gt_d)
    return front.reset_index(drop=True)

## Infill: use the inner emulated population

From the inner emulated Pareto population we choose `N_INFILL` members to spend true-model runs on, balancing two classic acquisition ideas:

- **exploit** - take points spread evenly along the emulated trade-off front, so we sharpen the front where the emulator already thinks it is;
- **explore** - take the points where the emulator is *most uncertain* (largest summed `*_gprstd`), so we sample where the surrogate is least trustworthy.

These true evaluations get added to the training set and the GPR is refit, closing the outer loop.

In [ ]:
def select_infill(front, n, cycle):
    f = front.sort_values(OBJS[0]).reset_index(drop=True)
    n_exploit = max(1, n // 2)
    # exploit: even spread along objective 1 of the emulated front
    idx = np.unique(np.linspace(0, f.shape[0]-1, n_exploit).round().astype(int))
    exploit = f.iloc[idx]
    # explore: largest total emulator predictive std
    stdcols = [o+"_gprstd" for o in OBJS if o+"_gprstd" in f.columns]
    remaining = f.drop(index=exploit.index)
    if stdcols and remaining.shape[0] > 0:
        norm = remaining[stdcols].apply(lambda c: (c-c.min())/(c.max()-c.min()+1e-30))
        score = norm.sum(axis=1).sort_values(ascending=False)
        explore = remaining.loc[score.index[:n-exploit.shape[0]]]
    else:
        explore = remaining.sample(min(n-exploit.shape[0], remaining.shape[0]),
                                    random_state=cycle)
    sel = pd.concat([exploit, explore]).drop_duplicates(subset=dvnames)
    return sel[dvnames].reset_index(drop=True), exploit, explore

## The outer Bayesian-optimization loop

Now we tie it together. For each of the `N_OUTER` cycles we: run the inner emulated MOU, pick infill points from that emulated population, evaluate them on the true model, append, and refit the GPR. We stash everything needed to visualize the process afterwards.

In [ ]:
def pareto_front(df):
    """non-dominated subset for two minimized objectives."""
    v = df[OBJS].values
    n = v.shape[0]
    dom = np.zeros(n, dtype=bool)
    for i in range(n):
        # i is dominated if some other member is <= in both objectives and < in one
        others = np.arange(n) != i
        dom[i] = np.any(others & np.all(v <= v[i], axis=1) & np.any(v < v[i], axis=1))
    return df.loc[~dom].sort_values(OBJS[0])

history = []
t0 = time.time()
for cycle in range(N_OUTER):
    print(f"\n=== outer cycle {cycle+1}/{N_OUTER} ===")
    front = inner_mou(gpr, cycle)                       # inner emulated MOU
    print(f"  emulated Pareto population: {front.shape[0]} members")
    infill_dv, exploit, explore = select_infill(front, N_INFILL, cycle)
    print(f"  selected {infill_dv.shape[0]} infill points; evaluating on true model...")
    new = evaluate_true(infill_dv, f"infill{cycle}")    # true-model infill
    new["cycle"] = cycle + 1
    history.append({"cycle": cycle+1, "front": front,
                    "infill": new.copy(), "train_before": train.copy(),
                    "n_exploit": exploit.shape[0]})
    train = pd.concat([train, new]).reset_index(drop=True)
    gpr = fit_gpr(train)                                # refit with augmented data
    print(f"  training set now {train.shape[0]}; true runs total {true_run_count['n']}")
print(f"\nouter loop done in {time.time()-t0:.0f}s; {true_run_count['n']} true-model runs used")

## A reference: full MOU on the true model

For comparison, run a "vanilla" PESTPP-MOU directly against the true model - the same thing the previous MOU notebook did, just sized modestly here. This gives us a reference Pareto front and a reference true-run count to judge the surrogate-assisted approach against.

In [ ]:
REF_POP, REF_GENS = 50, 20
pst_ref = pyemu.Pst(os.path.join(t_d,"pest_swp.pst"))
pst_ref.pestpp_options = {}
pst_ref.pestpp_options["mou_objectives"] = OBJS
pst_ref.pestpp_options["mou_population_size"] = REF_POP
pst_ref.pestpp_options["mou_save_population_every"] = REF_GENS
pst_ref.pestpp_options["mou_generator"] = "de"
pst_ref.observation_data.loc[OBJS,"weight"] = 1.0
pst_ref.observation_data.loc[OBJS,"obgnme"] = "less_than_obj"
pst_ref.control_data.noptmax = REF_GENS
pst_ref.write(os.path.join(t_d,"pest_ref.pst"), version=2)

ref_m_d = "master_mou_ref"
if os.path.exists(ref_m_d): shutil.rmtree(ref_m_d)
print(f"running reference full MOU ({REF_POP} x {REF_GENS} = {REF_POP*REF_GENS} true runs)...")
pyemu.os_utils.start_workers(t_d,"pestpp-mou","pest_ref.pst",num_workers=NUM_WORKERS,
                             worker_root=".", master_dir=ref_m_d)
ref_obs = pd.read_csv(os.path.join(ref_m_d,"pest_ref.archive.obs_pop.csv"), index_col=0)
ref_front = pareto_front(ref_obs[OBJS].copy())
ref_runs = REF_POP * REF_GENS
print(f"reference front: {ref_front.shape[0]} members from ~{ref_runs} true runs")

## Visualizing the inner-outer process

This is the payoff. For each outer cycle we plot objective space (both objectives minimized, so **down-and-left is better**) showing:

- **grey dots** - all true-model points evaluated *before* this cycle (the training set the emulator saw);
- **orange line** - the **emulated Pareto population** produced by the inner MOU on the GPR surrogate;
- **red = exploit / purple = explore stars** - the **infill** points chosen from that emulated population and then evaluated on the true model;
- **blue line** - the current true non-dominated front.

Watch the orange emulated front and the blue true front pull toward each other and toward the lower-left as cycles accumulate - that is the surrogate being corrected exactly where it matters.

In [ ]:
def plot_cycles(history):
    ncy = len(history)
    fig, axes = plt.subplots(1, ncy, figsize=(5*ncy, 4.5), squeeze=False)
    axes = axes[0]
    o0, o1 = OBJS
    for ax, h in zip(axes, history):
        tb = h["train_before"]; fr = h["front"]; nf = h["infill"]
        ne = h["n_exploit"]
        ax.scatter(tb[o0], tb[o1], s=14, c="0.6", alpha=0.5, label="train (before)")
        # the emulated archive can contain dominated members; show its clean front
        fr_pf = pareto_front(fr)
        ax.plot(fr_pf[o0], fr_pf[o1], "-", color="darkorange", lw=1.6, alpha=0.9, label="emulated front")
        ax.scatter(nf[o0].iloc[:ne], nf[o1].iloc[:ne], marker="*", s=120,
                   c="red", ec="k", lw=0.4, label="infill (exploit)", zorder=5)
        ax.scatter(nf[o0].iloc[ne:], nf[o1].iloc[ne:], marker="*", s=120,
                   c="purple", ec="k", lw=0.4, label="infill (explore)", zorder=5)
        cur = pd.concat([tb, nf])
        pf = pareto_front(cur)
        ax.plot(pf[o0], pf[o1], "-o", color="b", ms=3, lw=1.0, label="true Pareto front")
        ax.set_title(f"outer cycle {h['cycle']}")
        ax.set_xlabel(OBJ_LABELS[o0]); ax.set_ylabel(OBJ_LABELS[o1])
    axes[0].legend(fontsize=8, loc="upper right")
    fig.tight_layout()
    return fig

_ = plot_cycles(history)

### GPR-BO front vs the reference

Finally, overlay the Pareto front discovered by the surrogate-assisted search against the reference full-MOU front, and tally the true-model runs each used.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
o0, o1 = OBJS
ax.scatter(train[o0], train[o1], s=14, c="0.7", alpha=0.5, label=f"all GPR-BO true runs (n={true_run_count['n']})")
bo_front = pareto_front(train)
ax.plot(bo_front[o0], bo_front[o1], "-o", color="b", ms=4, lw=1.4,
        label="GPR-BO Pareto front")
ax.plot(ref_front[o0], ref_front[o1], "-s", color="green", ms=4, lw=1.4, alpha=0.8,
        label=f"reference full-MOU front (n~{ref_runs})")
ax.set_xlabel(OBJ_LABELS[o0]); ax.set_ylabel(OBJ_LABELS[o1])
ax.set_title("surrogate-assisted vs full MOU")
ax.legend(fontsize=9)
fig.tight_layout()

print(f"GPR-BO true-model runs : {true_run_count['n']}")
print(f"reference MOU true runs: ~{ref_runs}")
print(f"speed-up factor        : ~{ref_runs/max(1,true_run_count['n']):.1f}x fewer true runs")

# Final remarks

We just ran a **multi-objective Bayesian optimization** of the Freyberg management problem driven by a `pyemu` **GPR emulator**, in an **inner-outer** loop:

- the **outer** loop spends a small, targeted true-model budget and (re)trains the surrogate;
- the **inner** loop runs a complete PESTPP-MOU evolutionary search against the cheap emulator (in memory, via `gpr_pyworker`), producing an emulated Pareto population;
- **infill** points are drawn from that emulated population - half to sharpen the trade-off front (exploit), half where the emulator is most uncertain (explore) - and fed back to the true model.

Things worth remembering:
- **Standardize the objectives** before fitting the GP. With raw ~1e7 targets the surrogate is useless; standardized it generalizes well. This is the single most important practical detail here.
- **Dimensionality is the cost driver.** With ~84 decision variables the GP needs a reasonably sized initial design to generalize; in lower-dimensional problems you can start much smaller and lean harder on infill.
- **The inner search is essentially free**, so you can afford large emulated populations / many generations and a rich infill batch each cycle.
- **Use the emulator's uncertainty.** The `return_std` predictions are what make the "explore" half of infill principled rather than random - this is what makes it *Bayesian* optimization.
- Increase **`N_OUTER`** (and/or `N_INFILL`) to spend more true-model budget refining the front; the visualization makes it easy to see when extra cycles stop buying you much.